# Plot Az-constrained back-slip inversion — real-data GNSS interseismic locking

Counterpart to `plt_slip_inv_hetmu_nicoyaCK_locking_both.ipynb` for the Az-constrained scheme.

The Az scheme inverts a **scalar coupling amplitude** m(x) ∈ [0, amp_max] with
slip direction fixed to the plate-convergence azimuth (N33.5°E).  
Back-slip: `slip_vec = m_phys(x) × (c_strike(x), c_dip(x))`, where the coefficients
encode the in-plane back-slip direction (Savage 1983).

**Coupling shown**: `coupling = m_phys / amp_max ≈ ||slip|| / V_plate`  
(approximation: `||c|| ≈ 0.97–1.0` for Nicoya's 10–20° dip; see memory)

**`inv_str` format**: `_lockbothaz{az}_w{wh}{wv}_gs{gamma:.0e}_ds{delta:.0e}`  
**Output files**: `m_s_fault_*.txt` (2-col s_strike, s_dip), `moment_dip_*.txt` (6-col)


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'


In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/locking/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
flag_savefig = True
# flag_savefig = False

# Define the inversion results path
resultpath = "rst_locking/"

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
meshname = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface
use_uneven_mesh = "une" in meshname
print(meshname)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
lon0, lat0 = -84, 7     # from Christos's email
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] / 1e3

# Read the trench file
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)

type = "both"   # use both trench-parallel and normal components

# Import GNSS data from Kyriakopoulos et al. (2016) Figure 6
obs_disp_name = "data/CKfig6_data_final.csv"
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1,
                 names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car',
                        'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])
error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())


In [ ]:
# Az-constrained scheme.
# Amplitude transform mode (set by use_tanh_amp in the inversion script):
#   use_tanh_amp=True  -> bounded:   m_phys = amp_max * (tanh(m)+1)/2  in [0, amp_max]
#   use_tanh_amp=False -> unbounded: m_phys = m                         (linear, no bound)
# Filenames embed amp_tag = '' (bounded) or 'u' (unbounded) right after az_tag.

V_para_raw   = 27.0    # raw trench-parallel of Cocos-Caribbean motion, mm/yr
V_para_const = 11.0    # constant trench-parallel unrelated to tectonic loading
V_norm       = 78.5    # trench-normal of Cocos-Caribbean motion, mm/yr
V_para       = V_para_raw - V_para_const   # net trench-parallel
V_plate      = round(np.sqrt(V_norm**2 + V_para**2))   # full plate vector magnitude, mm/yr

# amp_max      = V_plate / 1e3   # amplitude bound in m/yr (matches inversion script)
# azimuth_deg  = 33.5    # geographic azimuth of plate convergence, CW from North
# az_tag       = "34"    # f"{33.5:.0f}" rounds to "34"; hardcoded to match inversion script filenames

amp_max      = V_norm / 1e3   # amplitude bound in m/yr (matches inversion script)
azimuth_deg  = 45    # geographic azimuth of plate convergence, CW from North
az_tag       = "45"    # f"{45:.0f}" rounds to "45"; hardcoded to match inversion script filenames

# Flip this to load the unbounded (linear) runs instead of the bounded (tanh) ones.
# Outputs from the two modes are kept distinct via the filename tag amp_tag.
use_tanh_amp = False
amp_tag      = '' if use_tanh_amp else 'u'

print(f"amp_max     = {amp_max*1e3:.1f} mm/yr")
print(f"az_tag      = {az_tag}")
print(f"Azimuth     = {azimuth_deg} deg CW from North")
print(f"amp mode    = {'tanh-bounded' if use_tanh_amp else 'linear-unbounded'}  (amp_tag={amp_tag!r})")

In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv"
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1,
                      names=['id', 'lat', 'lon', 'elv'])

region  = [-87, -84, 8.5, 11.5]
region1 = [-86.75, -84.4, 8.75, 11.25]

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]


In [ ]:
# Decide the weights of the horizontal, vertical components
f_h, f_v = 1, 1
print(f"Data weight horizontal / vertical: {f_h:.2f} / {f_v:.2f}")
w_h, w_v = int(1/f_h), int(1/f_v)

# Prior correlation length: rho = sqrt(gamma/delta) ~ 15 km (same as inversion script)
rho_s = 2.5e8

# Preferred gamma from L-curve analysis (plt_syn_slip_az_consistfwd_nicoya_locking_lc.ipynb).
# Het K2L corner ~ 3e2-5e2; 3D Het corner ~ 2e2-4e2.
# gamma_val_H1 = 3e2   # or 6e1, for bounded inversions
gamma_val_H1 = 4e3  # 4e3 seems the best for az45
# gamma_val_H1 = 4e3  # keep the same as az45 for az33.5, for consistency and comparison
delta_val_L2 = gamma_val_H1 / rho_s

# inv_str format matches slip_inv_hetmu_nicoyaCK_locking_both_az.py
inv_str = f"_lockbothaz{az_tag}{amp_tag}_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
print("inv_str:", inv_str)
print(f"gamma = {gamma_val_H1:.0e}, delta = {delta_val_L2:.1e}, rho_s = {rho_s:.1e}")

In [ ]:
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot
    return x_rot, y_rot

def read_obs_disp(datadir, resultpath, meshname, inv_str):
    outFileName = 'd_obs_' + meshname + inv_str + '.txt'
    print(outFileName)
    u_obs = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_obs['ux'], u_obs['uy'] = rot_xy(u_obs['ux'], u_obs['uy'], -rot)
    return u_obs

def build_disp_vector(df, u_obs, error_e, error_n, error_v):
    df_obs_h = pd.DataFrame(data={
        "x": df['lon'], "y": df['lat'],
        "east_velocity":  u_obs['ux']*1000,
        "north_velocity": u_obs['uy']*1000,
        "east_sigma":  error_e*1000,
        "north_sigma": error_n*1000,
        "correlation_EN": 0.0,
    })
    df_obs_v = pd.DataFrame(data={
        "x": df['lon'], "y": df['lat'],
        "east_velocity":  0.0,
        "north_velocity": u_obs['uz']*1000,
        "east_sigma":  error_v*1000,
        "north_sigma": error_v*1000,
        "correlation_EN": 0.0,
    })
    return df_obs_h, df_obs_v


In [ ]:
# read in the observation disp.
u_obs = read_obs_disp(datadir, resultpath, meshname, inv_str)

# build observation disp. vectors
df_obs_h, df_obs_v = build_disp_vector(df, u_obs, error_e, error_n, error_v)


In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

mu_b = 0;  mu_background = mu_expression(mu_b)
mu_l = 0.9730;  mu_lower = mu_expression(mu_l)   # ~55 GPa
mu_u = -0.9730; mu_upper = mu_expression(mu_u)   # ~25 GPa

# string for the homogeneous solution
homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
# string for the contrast between slab and wedge
sw_str   = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
# string for the 3D DeShon model (amplified ×2.5 relative 1D, CG2)
contrast_factor = 2.5
CG_mu_deg = 2
het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull_CG{CG_mu_deg}"
het3d_ori_str = f"_DeShon3D_ref_{round(1.0)}_hull_CG{CG_mu_deg}"
het1d_str = f"_DeShon1Dref"  # 1D depth-layered model

print("homo_str :", homo_str)
print("sw_str   :", sw_str)
print("het3d_str:", het3d_str)
print("het3d_ori_str:", het3d_ori_str)
print("het1d_str:", het1d_str)

In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
rot = 45  # rotation angle in degrees, positive is CCW
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m
km2m = 1e3

# Load the fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    skiprows=lambda x: x % 3 != 1,
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f / km2m
loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)
print(f'fault geometry: {len(loc_f)} rows')


In [ ]:
# Load per-vertex fault basis (mesh strike/dip unit vectors), row-aligned with m_s_fault.
fault_basis = pd.read_csv(datadir + resultpath + 'fault_basis_' + meshname + '.txt',
                          sep=r'\s+', comment='#',
                          names=['x_m', 'y_m', 'z_m',
                                 'strike_x', 'strike_y', 'strike_z',
                                 'dip_x', 'dip_y', 'dip_z'])
print(f'fault_basis: {len(fault_basis)} rows; should match m_s_* and loc_f row counts.')


In [ ]:
# Load the predicted surface displacement, all in meters
def read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_pred['ux'], u_pred['uy'] = rot_xy(u_pred['ux'], u_pred['uy'], -rot)
    u_res = u_pred.copy()
    u_res['ux'] = u_obs['ux'] - u_pred['ux']
    u_res['uy'] = u_obs['uy'] - u_pred['uy']
    u_res['uz'] = u_obs['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    return u_pred, u_res


def read_inferred_slip_az(datadir, resultpath, meshname, inv_str, struc_str_inv):
    """Read m_s_fault_*.txt (2-col: s_strike s_dip in m/yr) from Az inversion.
    Columns are m_phys*c_strike and m_phys*c_dip; mag = m_phys*||c|| ~ m_phys."""
    outFileName = 'm_s_fault_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                      names=['s_strike', 's_dip'])
    m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)
    m_s[['s_strike', 's_dip', 'mag']] *= 1e3   # m/yr -> mm/yr
    print(f"  s_strike: [{m_s['s_strike'].min():.3f}, {m_s['s_strike'].max():.3f}] mm/yr")
    print(f"  s_dip:    [{m_s['s_dip'].min():.3f}, {m_s['s_dip'].max():.3f}] mm/yr")
    print(f"  mag:      [{m_s['mag'].min():.3f}, {m_s['mag'].max():.3f}] mm/yr")
    return m_s


def read_pred_moment_az(datadir, resultpath, meshname, inv_str, struc_str_inv):
    """Read moment_dip_*.txt (6-col: moment mw potency moment_dip mw_dip potency_dip)."""
    outFileName = 'moment_dip_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                      names=['moment', 'mw', 'potency', 'moment_dip', 'mw_dip', 'potency_dip'])
    return mom


In [ ]:
##### Load the results of the homogeneous inversion
u_pred_hom, u_res_hom = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, homo_str)
m_s_hom = read_inferred_slip_az(datadir, resultpath, meshname, inv_str, homo_str)

##### Load the results of the slab/wedge (K_2LAYER het) inversion
u_pred_sw, u_res_sw = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, sw_str)
m_s_sw = read_inferred_slip_az(datadir, resultpath, meshname, inv_str, sw_str)

##### Load the results of the depth-layered 1d model inversion
u_pred_1d, u_res_1d = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, het1d_str)
m_s_1d = read_inferred_slip_az(datadir, resultpath, meshname, inv_str, het1d_str,)

##### Load the results of the original 3d model inversion
u_pred_3d_ori, u_res_3d_ori = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, het3d_ori_str)
m_s_3d_ori = read_inferred_slip_az(datadir, resultpath, meshname, inv_str, het3d_ori_str)

##### Load the results of the 3D heterogeneous (DeShon) inversion
u_pred_3d, u_res_3d = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, het3d_str)
m_s_3d = read_inferred_slip_az(datadir, resultpath, meshname, inv_str, het3d_str)


In [ ]:
mom_rate_hom = read_pred_moment_az(datadir, resultpath, meshname, inv_str, homo_str)
mom_rate_sw  = read_pred_moment_az(datadir, resultpath, meshname, inv_str, sw_str)
mom_rate_1d = read_pred_moment_az(datadir, resultpath, meshname, inv_str, het1d_str)
mom_rate_3d_ori = read_pred_moment_az(datadir, resultpath, meshname, inv_str, het3d_ori_str)
mom_rate_3d  = read_pred_moment_az(datadir, resultpath, meshname, inv_str, het3d_str)

n_yr = 2012 - 1978   # years of interseismic strain accumulation
print(n_yr)

mom_hom = mom_rate_hom['moment_dip'] * n_yr
mom_sw  = mom_rate_sw['moment_dip']  * n_yr
mom_1d = mom_rate_1d['moment_dip']*n_yr
mom_3d_ori = mom_rate_3d_ori['moment_dip']*n_yr
mom_3d  = mom_rate_3d['moment_dip']  * n_yr

_, _, M_w_hom = utp.moment2mag(mom_hom)
_, _, M_w_sw  = utp.moment2mag(mom_sw)
_, _, M_w_1d = utp.moment2mag(mom_1d)
_, _, M_w_3d_ori = utp.moment2mag(mom_3d_ori)
_, _, M_w_3d  = utp.moment2mag(mom_3d)

print(f"Hom : moment_dip={mom_hom.iloc[0]:.3e} N·m, Mw={M_w_hom[0]:.2f}")
print(f"Het : moment_dip={mom_sw.iloc[0]:.3e} N·m, Mw={M_w_sw[0]:.2f}")
print(f"1D  : moment_dip={mom_1d.iloc[0]:.3e} N·m, Mw={M_w_1d[0]:.2f}")
print(f"3D_ori  : moment_dip={mom_3d_ori.iloc[0]:.3e} N·m, Mw={M_w_3d_ori[0]:.2f}")
print(f"3D  : moment_dip={mom_3d.iloc[0]:.3e} N·m, Mw={M_w_3d[0]:.2f}")

In [ ]:
def plot_inferred_slip_panel_az(fig, m_s, mom_rate, title_str, panel_label,
                                region, slipsybsz, colormap, maxslip=80, slipstep=8,
                                show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)
    order = m_s['mag'].argsort()
    fig.plot(x=loc_f['lon'].iloc[order], y=loc_f['lat'].iloc[order],
             style=slipsybsz, fill=m_s['mag'].iloc[order], cmap=True)
    fig.text(text=f"Max. back-slip rate: {m_s['mag'].max():.2f} mm/yr", x=region[0]+0.05, y=region[-2]+0.1,
             font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Potency rate: {mom_rate['potency'].iloc[0]:.2e} m@+3@+/yr", x=region[0]+0.05,
             y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

    # --- Slip vector overlay using per-vertex fault basis ---
    # slip_xyz = m_strike * strike_hat + m_dip * dip_hat (mesh-frame components).
    # Mesh frame is rotated +rot CCW from geographic; rotate (x_mesh, y_mesh) back to (east, north).
    slip_x_mesh = m_s['s_strike'].values * fault_basis['strike_x'].values \
                + m_s['s_dip'].values    * fault_basis['dip_x'].values
    slip_y_mesh = m_s['s_strike'].values * fault_basis['strike_y'].values \
                + m_s['s_dip'].values    * fault_basis['dip_y'].values
    slip_e, slip_n = rot_xy(slip_x_mesh, slip_y_mesh, -rot)

    slip_grid_spacing = 0.2   # degrees, coarser than the slip dot grid for legibility
    bm_e = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=slip_e,
                             region=region, spacing=f"{slip_grid_spacing}d")
    bm_n = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=slip_n,
                             region=region, spacing=f"{slip_grid_spacing}d")
    sign = 1    # -1
    df_slip_vec = pd.DataFrame({
        'x':              bm_e.iloc[:, 0],
        'y':              bm_e.iloc[:, 1],
        'east_velocity':  bm_e.iloc[:, 2] * sign,
        'north_velocity': bm_n.iloc[:, 2] * sign,
    })

    scale_slip = amp_max*1e3   # mm/yr per scale-arrow unit
    fig.velo(data=df_slip_vec, pen="0.2p,black",
             spec=f"e{0.75/scale_slip}/0",
             vector="0.075c+a45+p0.1p,black+ea+gblack+h0")

    # Legend: net plate-motion vector V_plate at azimuth_deg (CW from N).
    # Drawn with the SAME scale and arrow style as the slip vectors above so
    # users can compare slip magnitude/orientation against the plate convergence.
    az_rad = np.deg2rad(azimuth_deg)
    print(f"Azimuth: {azimuth_deg:.1f} degrees")
    lgd = pd.DataFrame({
        'x': [region[0]+0.5], 'y': [region[-2]+0.35],
        'east_velocity':  [amp_max*1e3 * np.sin(az_rad)],
        'north_velocity': [amp_max*1e3 * np.cos(az_rad)],
    })
    fig.velo(data=lgd, pen="2p,black", spec=f"e{0.75/scale_slip}/0",
             vector="0.25c+a45+p1p,black+ea+gblack+h0")
    if azimuth_deg == 33.5:
        vplate_str = "V@-plate@-="
    elif azimuth_deg == 45:
        vplate_str = "V@-dip@-="
    fig.text(text=vplate_str,    
             x=region[0]+0.3, y=region[-2]+0.65,
             font="6p,Helvetica,black", justify="MC")
    fig.text(text=f"{amp_max*1e3:.1f} mm/yr",
             x=region[0]+0.3, y=region[-2]+0.5,
             font="6p,Helvetica,black", justify="MC")

    if show_cbar:
        colorlabel = "Back-slip rate"
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            # fig.colorbar(frame=["af", "x+lBack-slip rates", "y+l(mm/yr)"], position="JBC+h+o0/0.5c")
            fig.colorbar(frame=[f"a{slipstep}f{slipstep}", f"x+l{colorlabel}", "y+l(mm/yr)"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1,
             font="9p,Helvetica-Bold,black", justify="CM")


def plot_all_inferred_slip_az(m_s_hom, m_s_sw, m_s_1d, m_s_3d,
                               mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d,
                               region, flag_savefig=False):
    slipsybsz = "c0.11c"
    colormap  = "viridis"
    maxslip   = 80
    slipstep  = 8

    # mu_model_strings = ["H", "S", "3D"]
    mu_model_strings = ["H", "S", "1D", "Orig. 3D", "S - H", "1D - H", "Orig. 3D - H"]
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=4, figsize=("20c", "11c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.2c"], sharey=True):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_slip_panel_az(fig, m_s_hom, mom_rate_hom,
                                        mu_model_strings[0], panel_labels[0],
                                        region, slipsybsz, colormap, maxslip, slipstep)
        with fig.set_panel(panel=[0, 1]):
            plot_inferred_slip_panel_az(fig, m_s_sw, mom_rate_sw,
                                        mu_model_strings[1], panel_labels[1],
                                        region, slipsybsz, colormap, maxslip, slipstep)
        with fig.set_panel(panel=[0, 2]):
            plot_inferred_slip_panel_az(fig, m_s_1d, mom_rate_1d,
                                        mu_model_strings[2], panel_labels[2],
                                        region, slipsybsz, colormap, maxslip, slipstep)
        with fig.set_panel(panel=[0, 3]):
            plot_inferred_slip_panel_az(fig, m_s_3d, mom_rate_3d,
                                        mu_model_strings[3], panel_labels[3],
                                        region, slipsybsz, colormap, maxslip, slipstep)

    if flag_savefig:
        output_file = (figpath + f'backslip_az{az_tag}{amp_tag}_{meshname}.pdf')
        fig.savefig(output_file, dpi=300)
        print("Saved:", output_file)
    fig.show()


plot_all_inferred_slip_az(
    m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
    mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori,
    region1, flag_savefig=flag_savefig)

In [ ]:
# Az locking = mag_mmyr / V_plate_mmyr  (= m_phys*||c|| / amp_max ~ m_phys/amp_max)
# For Nicoya's 10-20 deg dip, ||c|| = 0.97-1.0 -> locking is within ~3% of true m_phys/amp_max.
def az_locking(m_s, V_plate_mmyr):
    m_s['locking'] = m_s['mag'] / V_plate_mmyr
    return m_s

m_s_hom    = az_locking(m_s_hom,    amp_max*1e3)
m_s_sw     = az_locking(m_s_sw,     amp_max*1e3)
m_s_1d     = az_locking(m_s_1d,     amp_max*1e3)
m_s_3d_ori = az_locking(m_s_3d_ori, amp_max*1e3)
m_s_3d     = az_locking(m_s_3d,     amp_max*1e3)

col2plt     = 'locking'
potency_col = 'potency'

print(f"Hom    locking: [{m_s_hom['locking'].min():.3f}, {m_s_hom['locking'].max():.3f}]")
print(f"Het S  locking: [{m_s_sw['locking'].min():.3f}, {m_s_sw['locking'].max():.3f}]")
print(f"1D     locking: [{m_s_1d['locking'].min():.3f}, {m_s_1d['locking'].max():.3f}]")
print(f"3D-ori locking: [{m_s_3d_ori['locking'].min():.3f}, {m_s_3d_ori['locking'].max():.3f}]")
print(f"3D     locking: [{m_s_3d['locking'].min():.3f}, {m_s_3d['locking'].max():.3f}]")


In [ ]:
def plot_inferred_locking_panel(fig, m_s, mom_rate, panel_idx, title_str, panel_label,
                                region, col2plt, slipsybsz, colormap, show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    maxslip = 1
    slipstep = 0.1
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s[col2plt], region=region, spacing=(0.05, 0.05))
    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"Max. locking: {m_s[col2plt].max():.2f}", x=region[0]+0.05, y=region[-2]+0.1,
             font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Potency rate: {mom_rate[potency_col].iloc[0]:.2e} m@+3@+/yr", x=region[0]+0.05,
             y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    if panel_idx == 0:
        x0, y0 = region[0]+0.05, region[-2]+0.35
        width = 0.7
        height = 0.35
        fig.plot(x=[x0, x0+width, x0+width, x0, x0], y=[y0, y0, y0+height, y0+height, y0],
                 pen="0.5p,black", fill="lightgray")
        x_legend = region[0] + 0.1
        y_legend = region[2] + 0.45
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15], pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend], pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking", font="5p,Helvetica", justify="LM")

In [ ]:
def plot_inferred_locking_diff_panel(fig, m_s, mom_rate, title_str, panel_label, maxdiff, diff_step,
                                     region, col2plt, slipsybsz, colormap, show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
    m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
    order = m_s_diff.argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s[col2plt], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"@~D@~ Locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
             x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    diff = mom_rate[potency_col].iloc[0] - mom_rate_hom[potency_col].iloc[0]
    fig.text(text=f"@~D@~ Potency rate: {diff:.2e} m@+3@+/yr",
             x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lDifference in locking ratio"],
                         position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

In [ ]:
def plot_all_inferred_locking_and_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
                                       mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori,
                                       col2plt, region, maxdiff=0.3, diff_step=0.05,
                                       flag_savefig=False):
    slipsybsz     = "c0.11c"
    colormap      = "viridis"
    colormap_diff = "roma"

    mu_model_strings = ["H", "S", "1D", "Orig. 3D",
                        "S - H", "1D - H", "Orig. 3D - H"]
    panel_labels     = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=4, figsize=("20c", "11.0c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.0c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_locking_panel(fig, m_s_hom, mom_rate_hom, panel_idx,
                                        mu_model_strings[panel_idx], panel_labels[panel_idx],
                                        region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_sw, mom_rate_sw, panel_idx,
                                        mu_model_strings[panel_idx], panel_labels[panel_idx],
                                        region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 2]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_1d, mom_rate_1d, panel_idx,
                                        mu_model_strings[panel_idx], panel_labels[panel_idx],
                                        region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 3]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_3d_ori, mom_rate_3d_ori, panel_idx,
                                        mu_model_strings[panel_idx], panel_labels[panel_idx],
                                        region, col2plt, slipsybsz, colormap, show_cbar=False)

        # Empty panel [1,0]: two shared colorbars (locking + diff)
        with fig.set_panel(panel=[1, 0]):
            with pygmt.config(MAP_FRAME_PEN="0p,white", MAP_TICK_PEN_PRIMARY="0p,white"):
                fig.basemap(region=[0, 1, 0, 1], projection="X?/?", frame="0")
            pygmt.makecpt(cmap=colormap, series=[0, 1, 0.1], background=True, reverse=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=["a0.2f0.1", "x+lInterseismic locking ratio"],
                             position="JMC+w3.8c/0.2c+h+o0c/2.4c")
            pygmt.makecpt(cmap=colormap_diff, series=[-maxdiff, maxdiff, diff_step],
                          background=True, reverse=False)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lDifference in locking ratio"],
                             position="JMC+w3.8c/0.2c+h+o0c/1.2c")

        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_sw, mom_rate_sw,
                                             mu_model_strings[panel_idx], panel_labels[panel_idx],
                                             maxdiff, diff_step,
                                             region, col2plt, slipsybsz, colormap_diff, show_cbar=False)
        with fig.set_panel(panel=[1, 2]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_1d, mom_rate_1d,
                                             mu_model_strings[panel_idx], panel_labels[panel_idx],
                                             maxdiff, diff_step,
                                             region, col2plt, slipsybsz, colormap_diff, show_cbar=False)
        with fig.set_panel(panel=[1, 3]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_3d_ori, mom_rate_3d_ori,
                                             mu_model_strings[panel_idx], panel_labels[panel_idx],
                                             maxdiff, diff_step,
                                             region, col2plt, slipsybsz, colormap_diff, show_cbar=False)

    if flag_savefig:
        output_file = (figpath + f'coupling_az{az_tag}{amp_tag}_rs{rho_s:.0e}_'
                       f'gs{gamma_val_H1:.0e}_{meshname}.pdf')
        fig.savefig(output_file, dpi=300)
        print("Saved:", output_file)
    fig.show()


In [ ]:
maxdiff = 0.14    
# diff_step = 0.02
diff_step = maxdiff/10

plot_all_inferred_locking_and_diff(
    m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
    mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori,
    col2plt, region1, maxdiff, diff_step, flag_savefig=False)


In [ ]:
# Supplementary: amplified 3D model (m_s_3d, contrast_factor = 2.5) — 2-panel
# layout with absolute locking and difference vs the homogeneous reference.
def plot_all_inferred_locking_and_diff_extra(m_s_3d, mom_rate_3d,
                                              col2plt, region, maxdiff, diff_step,
                                              flag_savefig=False):
    slipsybsz     = "c0.11c"
    colormap      = "viridis"
    colormap_diff = "roma"

    mu_model_strings = ["Ampl. 3D", "Ampl. 3D - H"]
    panel_labels     = ["(a)", "(b)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("10c", "12.5c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.25c"], sharey=True):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_locking_panel(fig, m_s_3d, mom_rate_3d, panel_idx,
                                        mu_model_strings[panel_idx], panel_labels[panel_idx],
                                        region, col2plt, slipsybsz, colormap)

        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_3d, mom_rate_3d,
                                             mu_model_strings[panel_idx], panel_labels[panel_idx],
                                             maxdiff, diff_step,
                                             region, col2plt, slipsybsz, colormap_diff)

    if flag_savefig:
        output_file = (figpath + f'coupling_az{az_tag}{amp_tag}_rs{rho_s:.0e}_'
                       f'gs{gamma_val_H1:.0e}_{meshname}_ampl3D.pdf')
        fig.savefig(output_file, dpi=300)
        print("Saved:", output_file)
    fig.show()


maxdiff = 0.14    
# diff_step = 0.02
diff_step = maxdiff/10

plot_all_inferred_locking_and_diff_extra(
    m_s_3d, mom_rate_3d, col2plt, region1, maxdiff=maxdiff, diff_step=diff_step,
    flag_savefig=False)


In [ ]:
# compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_obs) * 1000   # mm
rmse_sw  = utp.rmse_3d_vec_dataframe(u_pred_sw,  u_obs) * 1000
rmse_1d  = utp.rmse_3d_vec_dataframe(u_pred_1d,  u_obs) * 1000
rmse_3d_ori  = utp.rmse_3d_vec_dataframe(u_pred_3d_ori,  u_obs) * 1000
rmse_3d  = utp.rmse_3d_vec_dataframe(u_pred_3d,  u_obs) * 1000
print(f"RMSE (mm): Hom={rmse_hom:.3f}, Het={rmse_sw:.3f}, 1D={rmse_1d:.3f},",
      f"3D-Ori={rmse_3d_ori:.3f}, 3D={rmse_3d:.3f}")


In [ ]:
def build_pred_disp_vector(df, u_pred, u_res, error_e, error_n, error_v):
    df_pred_h = pd.DataFrame(data={
        "x": df['lon'], "y": df['lat'],
        "east_velocity":  u_pred['ux']*1000,
        "north_velocity": u_pred['uy']*1000,
    })
    df_pred_v = pd.DataFrame(data={
        "x": df['lon'], "y": df['lat'],
        "east_velocity":  0.0,
        "north_velocity": u_pred['uz']*1000,
    })
    df_res_h = pd.DataFrame(data={
        "x": df['lon'], "y": df['lat'],
        "east_velocity":  u_res['ux']*1000,
        "north_velocity": u_res['uy']*1000,
        "east_sigma":  error_e*1000,
        "north_sigma": error_n*1000,
        "correlation_EN": 0.0,
    })
    df_res_h_dum = df_res_h.iloc[:, :4]
    df_res_v = pd.DataFrame(data={
        "x": df['lon'], "y": df['lat'],
        "east_velocity":  0.0,
        "north_velocity": u_res['uz']*1000,
        "east_sigma":  error_v*1000,
        "north_sigma": error_v*1000,
        "correlation_EN": 0.0,
    })
    df_res_v_dum = df_res_v.iloc[:, :4]
    return df_pred_h, df_pred_v, df_res_h, df_res_h_dum, df_res_v, df_res_v_dum


In [ ]:
df_pred_hom_h, df_pred_hom_v, df_res_hom_h, df_res_hom_h_dum, df_res_hom_v, df_res_hom_v_dum = \
    build_pred_disp_vector(df, u_pred_hom, u_res_hom, error_e, error_n, error_v)

df_pred_sw_h, df_pred_sw_v, df_res_sw_h, df_res_sw_h_dum, df_res_sw_v, df_res_sw_v_dum = \
    build_pred_disp_vector(df, u_pred_sw, u_res_sw, error_e, error_n, error_v)

df_pred_1d_h, df_pred_1d_v, df_res_1d_h, df_res_1d_h_dum, df_res_1d_v, df_res_1d_v_dum = \
    build_pred_disp_vector(df, u_pred_1d, u_res_1d, error_e, error_n, error_v)

df_pred_3d_ori_h, df_pred_3d_ori_v, df_res_3d_ori_h, df_res_3d_ori_h_dum, df_res_3d_ori_v, df_res_3d_ori_v_dum = \
    build_pred_disp_vector(df, u_pred_3d_ori, u_res_3d_ori, error_e, error_n, error_v)

df_pred_3d_h, df_pred_3d_v, df_res_3d_h, df_res_3d_h_dum, df_res_3d_v, df_res_3d_v_dum = \
    build_pred_disp_vector(df, u_pred_3d, u_res_3d, error_e, error_n, error_v)


In [ ]:
def plot_disp_obs_pred_res(df_obs_h, df_pred_hom_h, df_pred_h,
                           df_res_hom_h_dum, df_res_h_dum,
                           df_obs_v, df_pred_hom_v, df_pred_v,
                           df_res_hom_v_dum, df_res_v_dum,
                           u_obs, u_pred_hom, u_pred, struc_str_inv,
                           region, flag_savefig=False):
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=3, figsize=("18c", "12c"), autolabel=False,
                     frame=["a1f0.5", "WSne"], margins=["0.1c", "0.2c"], sharey=True):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"

        # horizontal Hom pred vs obs
        with fig.set_panel(panel=[0, 0]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0})
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # horizontal Het pred vs obs
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0})
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # horizontal residuals
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.velo(data=df_res_hom_h_dum, pen="0.5p,blue", line="0.25p,blue",
                     spec="e0.16/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.velo(data=df_res_h_dum, pen="0.5p,red", line="0.25p,red",
                     spec="e0.16/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_obs) * 1000
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse     = utp.rmse_3d_vec_dataframe(u_pred,     u_obs) * 1000
            fig.text(text=f"RMSE: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.2,
                     font="9p,Helvetica,blue", justify="ML")
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.4,
                     font="9p,Helvetica,red", justify="ML")
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # vertical Hom pred vs obs
        with fig.set_panel(panel=[1, 0]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            fig.velo(data=df_pred_hom_v, pen="0.5p,blue", line="0.25p,blue",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0})
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # vertical Het pred vs obs
        with fig.set_panel(panel=[1, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            fig.velo(data=df_pred_v, pen="0.5p,red", line="0.25p,red",
                     spec="e0.05/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0})
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # vertical residuals
        with fig.set_panel(panel=[1, 2]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05))
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.velo(data=df_res_hom_v_dum, pen="0.5p,blue", line="0.25p,blue",
                     spec="e0.16/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.velo(data=df_res_v_dum, pen="0.5p,red", line="0.25p,red",
                     spec="e0.16/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_obs) * 1000
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse     = utp.rmse_3d_vec_dataframe(u_pred,     u_obs) * 1000
            fig.text(text=f"RMSE: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.2,
                     font="9p,Helvetica,blue", justify="ML")
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.4,
                     font="9p,Helvetica,red", justify="ML")
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            lgdres = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    fig.show()

    if flag_savefig:
        output_file = (figpath + f'disp_fit_az{az_tag}_rs{rho_s:.0e}_'
                       f'gs{gamma_val_H1:.0e}_{meshname}{struc_str_inv}.pdf')
        fig.savefig(output_file, dpi=300)
        print("Saved:", output_file)

In [ ]:
# plot_disp_obs_pred_res(
#     df_obs_h, df_pred_hom_h, df_pred_sw_h,
#     df_res_hom_h_dum, df_res_sw_h_dum,
#     df_obs_v, df_pred_hom_v, df_pred_sw_v,
#     df_res_hom_v_dum, df_res_sw_v_dum,
#     u_obs, u_pred_hom, u_pred_sw, sw_str,
#     region1, flag_savefig=flag_savefig)

# swap to show 3D Het fit:
plot_disp_obs_pred_res(df_obs_h, df_pred_hom_h, df_pred_3d_ori_h, 
                       df_res_hom_h_dum, df_res_3d_ori_h_dum,
                       df_obs_v, df_pred_hom_v, df_pred_3d_ori_v, 
                       df_res_hom_v_dum, df_res_3d_ori_v_dum,
                       u_obs, u_pred_hom, u_pred_3d_ori, het3d_str, 
                       region1, flag_savefig=False)

In [ ]:
# Rose diagram: distribution of horizontal residual vector azimuths
# Residual = obs - pred. Reference lines: prescribed back-slip azimuth and its
# anti-parallel (same red solid line, one combined legend entry); per-panel
# weighted-mean residual direction (black dashed) is also drawn so the bias
# direction of each model can be read directly off the legend.

bin_deg = 5
bins        = np.arange(0, 361, bin_deg)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
theta = np.deg2rad(bin_centers)
width = np.deg2rad(bin_deg)

labels  = ["H", "S", "1D", "Orig. 3D", "Ampl. 3D"]
res_dfs = [df_res_hom_h, df_res_sw_h, df_res_1d_h, df_res_3d_ori_h, df_res_3d_h]

# Choose how to weight each residual:
#   'equal' → every station counts as 1 (pure directional distribution)
#   'amp'   → weight by |res| (mm/yr) so larger residuals dominate
weight_mode = 'amp'        # 'equal' or 'amp'

# Pre-compute weighted counts, RMSE, weighted-mean direction (and median + R for
# diagnostic printing) so we can set a shared radial limit across panels.
all_counts = []
all_az     = []
all_amp    = []
all_rmse   = []
all_mean   = []
all_median = []
all_R      = []
for df_res in res_dfs:
    e, n = df_res['east_velocity'].values, df_res['north_velocity'].values
    az_deg = np.degrees(np.arctan2(e, n)) % 360
    amp    = np.hypot(e, n)                       # raw 2D horizontal residual magnitude (mm/yr)
    rmse   = np.sqrt(np.mean(amp**2))             # per-vector RMSE on horizontal only

    if weight_mode == 'equal':
        weights = np.ones_like(amp)
    elif weight_mode == 'amp':
        weights = amp
    else:
        raise ValueError(f"unknown weight_mode: {weight_mode!r}")

    # weighted circular mean and R
    sin_s = np.sum(weights * np.sin(np.deg2rad(az_deg)))
    cos_s = np.sum(weights * np.cos(np.deg2rad(az_deg)))
    az_mean = np.degrees(np.arctan2(sin_s, cos_s)) % 360
    R       = np.hypot(sin_s, cos_s) / np.sum(weights)

    # weighted circular median (wrap to half-circle around the mean, then linear)
    az_centered = ((az_deg - az_mean + 180) % 360) - 180
    order_c     = np.argsort(az_centered)
    cum_w       = np.cumsum(weights[order_c]) / np.sum(weights)
    med_idx     = np.searchsorted(cum_w, 0.5)
    az_median   = (az_centered[order_c][med_idx] + az_mean) % 360

    counts, _ = np.histogram(az_deg, bins=bins, weights=weights)
    all_counts.append(counts)
    all_az.append(az_deg)
    all_amp.append(amp)
    all_rmse.append(rmse)
    all_mean.append(az_mean)
    all_median.append(az_median)
    all_R.append(R)

rmax = np.ceil(max(c.max() for c in all_counts) * 1.05)  # 5% headroom
print(f"rmax: {rmax:.1f}")

fig_rose, axes = plt.subplots(2, 4, figsize=(12, 8), dpi=300,
                               subplot_kw=dict(projection='polar'))
axes = axes.ravel()
for i, (ax, counts, az_deg, rmse, az_median, label) in enumerate(zip(
        axes[:5], all_counts, all_az, all_rmse, all_median, labels)):
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)   # clockwise = geographic convention
    ax.bar(theta, counts, width=width, bottom=0,
           color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_ylim(0, rmax)         # uniform across panels
    # Move the radial tick labels (5, 10, 15, 20, 25) along the NW direction,
    # clear of the rose bars in the NE quadrant.
    ax.set_rlabel_position(315)

    # Reference lines: prescribed back-slip azimuth + anti-parallel (same line
    # style, one combined legend entry) + per-panel weighted median direction.
    # Median is preferred over mean here because R is moderate (~0.5) — the
    # residual distributions are skewed, so the mean is pulled away from the
    # visual mode of the rose; the median tracks the bulk of the weight.
    h_pres, = ax.plot([], [], color='red', lw=1.5, linestyle='-',
                       label=f"prescribed {azimuth_deg:.1f}° / {(azimuth_deg + 180) % 360:.1f}°")
    h_med,  = ax.plot([], [], color='k', lw=1.8, linestyle='--',
                       label="weighted median")
    ax.axvline(np.deg2rad(azimuth_deg % 360),         color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad((azimuth_deg + 180) % 360), color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad(az_median), color='k', lw=1.8, linestyle='--')

    # Annotate the median angle value just CCW (toward N) of its line, mid-radius,
    # so it sits in an empty area off the rose bars in the NE quadrant.
    ax.text(np.deg2rad(az_median - 15),
            rmax * 0.8,
            f"{az_median:.1f}°",
            ha='center', va='center',
            fontsize=9, color='k')

    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    ax.set_title(f"{label}  (RMSE$_h$={rmse:.2f} mm/yr)", va='bottom', pad=12, fontsize=10)

    # Only the first panel carries the legend (saves space; entries are the
    # same across panels except for the angle values, which are now annotated
    # directly on each panel beside the line).
    if i == 0:
        ax.legend(handles=[h_pres, h_med],
                  loc='lower left', bbox_to_anchor=(-0.15, -0.30),
                  fontsize=9, frameon=False)

# Hide all trailing unused panels
for ax in axes[len(labels):]:
    ax.axis('off')

plt.suptitle(f"Horizontal residual azimuths  (obs − pred)  "
             f"(n={len(all_az[0])}, bin={bin_deg}°)",
             y=0.97, fontsize=11)
plt.tight_layout()
# if flag_savefig:
#     fig_rose.savefig(figpath + f'residual_az_rose_az{az_tag}{amp_tag}_gs{gamma_val_H1:.0e}_{weight_mode}_{meshname}.pdf',
#                      bbox_inches='tight')
plt.show()

# Diagnostic print: per-model R + weighted mean & median
print(f"weight_mode = {weight_mode!r}")
for lbl, az_m, az_md, R in zip(labels, all_mean, all_median, all_R):
    print(f"  {lbl:>10s}: mean={az_m:6.2f}°  median={az_md:6.2f}°  R={R:.3f}")
print(f"Prescribed back-slip azimuth: {azimuth_deg:.2f}° / anti-parallel {(azimuth_deg + 180) % 360:.2f}°")


In [ ]:
# Production version of the residual rose: only the 4 main μ models
# (H, S, 1D, Orig. 3D) in a single row.  Re-uses the per-model statistics
# already computed in the cell above (all_counts, all_az, all_rmse, all_median,
# rmax, labels), so this cell must run after the 5-panel reference cell.
# Ampl. 3D goes into its own separate figure below.

n_main  = 4
labels_m = labels[:n_main]

# Recompute rmax from the (already-populated) all_counts so this cell is
# idempotent — i.e. yields the same figure regardless of whether other cells
# (e.g. the 1x2 obs rose below) have been run since cell 26 and overwritten
# the module-level `rmax`.
rmax = np.ceil(max(c.max() for c in all_counts) * 1.05)  # 5% headroom
print(f"rmax: {rmax:.1f}")

fig_rose, axes = plt.subplots(1, n_main, figsize=(12, 3.6), dpi=300,
                               subplot_kw=dict(projection='polar'))
for i, (ax, counts, az_deg, rmse, az_median, label) in enumerate(zip(
        axes, all_counts[:n_main], all_az[:n_main],
        all_rmse[:n_main], all_median[:n_main], labels_m)):
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)   # clockwise = geographic convention
    ax.bar(theta, counts, width=width, bottom=0,
           color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_ylim(0, rmax)         # uniform across panels
    ax.set_rlabel_position(315)  # radial tick labels along NW

    # Reference lines: prescribed back-slip azimuth + anti-parallel (red solid,
    # one combined legend entry on panel 0) + per-panel weighted median (black dashed).
    h_pres, = ax.plot([], [], color='red', lw=1.5, linestyle='-',
                       label=f"prescribed {azimuth_deg:.1f}° / {(azimuth_deg + 180) % 360:.1f}°")
    h_med,  = ax.plot([], [], color='k', lw=1.8, linestyle='--',
                       label="weighted median")
    ax.axvline(np.deg2rad(azimuth_deg % 360),         color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad((azimuth_deg + 180) % 360), color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad(az_median), color='k', lw=1.8, linestyle='--')

    # Annotate median angle just CCW (toward N) of its line, mid-radius.
    ax.text(np.deg2rad(az_median - 15), rmax * 0.8,
            f"{az_median:.1f}°",
            ha='center', va='center',
            fontsize=9, color='k')

    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    ax.set_title(f"{label}  (RMSE$_h$={rmse:.2f} mm/yr)", va='bottom', pad=25, fontsize=10)

    if i == 0:
        ax.legend(handles=[h_pres, h_med],
                  loc='lower left', bbox_to_anchor=(-0.10, -0.30),
                  fontsize=9, frameon=False)

plt.suptitle(f"Azimuth-constrained (N{azimuth_deg}°E) inversion: Horizontal residual azimuths (obs − pred) "
             f"(n={len(all_az[0])}, bin={bin_deg}°)",
             y=1.02, fontsize=11)
plt.tight_layout()
if flag_savefig:
    fig_rose.savefig(figpath + f'residual_az_rose_az{az_tag}{amp_tag}_{meshname}.pdf',
                     bbox_inches='tight')
    print(f"Saved: {figpath + f'residual_az_rose_az{az_tag}{amp_tag}_{meshname}.pdf'}")
plt.show()


In [ ]:
# ----- Data-only rose: azimuths of GPS observation vectors (no inversion) -----
# Independent of any model parameterization. Tells us the dominant direction
# of horizontal GPS surface motion in the array.
#
# R is the resultant length, in [0, 1]:
#   R = || Σ wi (sin θi, cos θi) || / Σ wi
#   R = 1  →  all azimuths perfectly aligned (zero circular variance)
#   R = 0  →  uniform distribution around the circle (no preferred direction)
# A high R justifies summarising the field by a single mean direction.
#
# Two-panel comparison of weighting choices:
#   'equal' → every station counts as 1 (pure directional distribution)
#   'amp'   → weight by horizontal observed speed |obs| (mm/yr) so larger
#             motions dominate (matches the residual-rose convention)

e_obs   = df_obs_h['east_velocity'].values
n_obs   = df_obs_h['north_velocity'].values
az_obs  = np.degrees(np.arctan2(e_obs, n_obs)) % 360
amp_obs = np.hypot(e_obs, n_obs)

bin_deg = 5
bins        = np.arange(0, 361, bin_deg)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
theta = np.deg2rad(bin_centers); width = np.deg2rad(bin_deg)

# Pre-compute stats for both weight modes
weight_modes = ['equal', 'amp']
stats = {}
for wm in weight_modes:
    weights = np.ones_like(amp_obs) if wm == 'equal' else amp_obs

    sin_sum = np.sum(weights * np.sin(np.deg2rad(az_obs)))
    cos_sum = np.sum(weights * np.cos(np.deg2rad(az_obs)))
    az_mean   = np.degrees(np.arctan2(sin_sum, cos_sum)) % 360
    R         = np.hypot(sin_sum, cos_sum) / np.sum(weights)

    az_centered = ((az_obs - az_mean + 180) % 360) - 180
    order_c     = np.argsort(az_centered)
    cum_w       = np.cumsum(weights[order_c]) / np.sum(weights)
    med_idx     = np.searchsorted(cum_w, 0.5)
    az_median   = (az_centered[order_c][med_idx] + az_mean) % 360

    counts, _ = np.histogram(az_obs, bins=bins, weights=weights)
    stats[wm] = dict(counts=counts, az_mean=az_mean, az_median=az_median, R=R)

fig_obs, axes = plt.subplots(1, 2, figsize=(8, 5), dpi=300,
                              subplot_kw=dict(projection='polar'))

for i, (ax, wm) in enumerate(zip(axes, weight_modes)):
    s = stats[wm]
    rmax = np.ceil(s['counts'].max() * 1.05)  # 5% headroom

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.bar(theta, s['counts'], width=width, bottom=0,
           color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_ylim(0, rmax)
    ax.set_rlabel_position(315)   # radial tick labels along NW

    # Reference lines: prescribed back-slip azimuth + anti-parallel (red solid,
    # one combined legend entry) + weighted median (black dashed).
    h_pres, = ax.plot([], [], color='red', lw=1.5, linestyle='-',
                       label=f"prescribed {azimuth_deg:.1f}° / {(azimuth_deg + 180) % 360:.1f}°")
    h_med,  = ax.plot([], [], color='k', lw=1.8, linestyle='--',
                       label="weighted median")
    ax.axvline(np.deg2rad(azimuth_deg % 360),         color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad((azimuth_deg + 180) % 360), color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad(s['az_median']), color='k', lw=1.8, linestyle='--')

    # Annotate median angle just CCW of its line, mid-radius (same convention
    # as the residual-rose panels).
    ax.text(np.deg2rad(s['az_median'] - 10), rmax * 0.8,
            f"{s['az_median']:.1f}°",
            ha='center', va='center',
            fontsize=9, color='k')

    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    title_str = 'equal weight of 1' if wm == 'equal' else 'weighted by amp'
    ax.set_title(title_str, va='bottom', pad=12, fontsize=10)

    if i == 0:
        ax.legend(handles=[h_pres, h_med],
                  loc='lower left', bbox_to_anchor=(0.2, 0.05),
                  fontsize=9, frameon=False)

plt.suptitle(f"GNSS velocity azimuths (n={len(az_obs)}, bin={bin_deg}°)",
             y=0.97, fontsize=11)
plt.tight_layout()
if flag_savefig:
    fig_obs.savefig(figpath + f'obs_az_rose_{meshname}.pdf', bbox_inches='tight')
    print(f"Saved: {figpath + f'obs_az_rose_{meshname}.pdf'}")
plt.show()

# Diagnostic print
for wm in weight_modes:
    s = stats[wm]
    print(f"  weight={wm!r:<7s}: mean={s['az_mean']:6.2f}°  median={s['az_median']:6.2f}°  R={s['R']:.3f}")
print(f"Prescribed back-slip azimuth in this run: {azimuth_deg:.2f}° / anti-parallel {(azimuth_deg + 180) % 360:.2f}°")
